# 뉴스링고 (Newslingo) — 코드 워크스루

> 6반 5조 AI Agent 설계서 구현체 · LangChain 1.x (`create_agent` + middleware + Structured Output)

이 노트북은 **발표/코드리뷰용 설명서**입니다. 실제 로직은 `newslingo/` 패키지에 있고,
여기서는 설계서 섹션 순서대로 각 조각을 하나씩 불러와 확인합니다.

| 노트북 섹션 | 파일 | 설계서 |
|---|---|---|
| 1. Structured Output | `schemas.py` | 2.4 |
| 2. 모델1 / 모델2 | `models.py` | 2.3 |
| 3. 입력 가드레일 2단계 | `guardrails.py` | 3.3 |
| 4. Tool | `tools.py` | 2.5 |
| 5. Middleware | `middleware.py` | 3.2 |
| 6. Agent 조립 | `agent.py` | 2.1 |
| 7. LCEL 체인 | `chains.py` | 2.4 |
| 8. 채점 & 난이도 추천 | `grading.py` | 2.2 |
| 9. 전체 플로우 | `service.py` | 2.2 |


## 0. 준비

- `newslingo/.env` 에 `OPENAI_API_KEY` 를 넣어주세요.
- `NEWS_API_KEY` 가 없으면 `.env` 에 `NEWSLINGO_USE_MOCK_NEWS=true` (샘플 기사로 동작).


In [ ]:
import sys, pathlib
# 이 노트북(newslingo/notebooks/)에서 두 단계 위(=0909-0911_langchain)를 경로에 추가해
#  가 되게 한다.
ROOT = pathlib.Path.cwd()
while ROOT.name and not (ROOT / 'newslingo' / '__init__.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('project root:', ROOT)

from newslingo import config
print('메인 모델 :', config.MAIN_MODEL, '| temp', config.MAIN_TEMPERATURE)
print('분류 모델 :', config.CLASSIFIER_MODEL, '| temp', config.CLASSIFIER_TEMPERATURE)
print('난이도 임계치 : 상향', config.LEVEL_UP_THRESHOLD, '/ 하향', config.LEVEL_DOWN_THRESHOLD)
print('뉴스 목 모드 :', config.USE_MOCK_NEWS)

## 1. Structured Output (설계서 2.4)

강의 [3] LangChain *4. Structured Output* 의 Pydantic 방식. 모든 생성 결과의 '규격'을 미리 못박아 둡니다.


In [ ]:
from newslingo import schemas

# 설계서 2.4 표의 스키마들
for name in ['TopicGuardrailResult','ArticleCandidate','ArticleCandidateList',
             'ArticleStudyMaterial','QuizQuestion','QuizSet']:
    model = getattr(schemas, name)
    print(f'{name:22s}:', list(model.model_fields))

In [ ]:
# 예: 퀴즈 문항 스키마 상세
schemas.QuizQuestion.model_json_schema()

## 2. 모델1 / 모델2 (설계서 2.3)

- **모델1 (메인)**: gpt-4o-mini, temp 0.4 — 기사 추천·설명·채팅·퀴즈 생성
- **모델2 (분류)**: gpt-4o-mini, temp 0 — 입력 가드레일 전용 (판별 일관성)


In [ ]:
from newslingo.models import get_main_model, get_classifier_model

main = get_main_model()
print(main.invoke('Say hi in Korean, one short sentence.').content)

## 3. 입력 가드레일 2단계 (설계서 3.3)

저비용 **규칙 필터**를 먼저 → 통과분만 **모델2**로. 강의 [5] *3. Guardrails* 패턴.


In [ ]:
from newslingo.guardrails import rule_based_filter, model_based_filter, check_topic

# 1차: 금칙어 → 모델 호출 없이 즉시 차단
print('1차 차단  :', rule_based_filter('야한 뉴스 보여줘'))
print('1차 통과  :', rule_based_filter('반도체 뉴스로 공부하고 싶어'))

In [ ]:
# 2차: 의미 판별 (모델2 + TopicGuardrailResult)
print('정상   :', check_topic('요즘 우주 탐사 뉴스로 영어 공부하고 싶어'))
print('오프토픽:', check_topic('오늘 점심 뭐 먹을지 골라줘'))

## 4. Tool (설계서 2.5)

- `news_search` : NewsAPI.org 검색 (실패 시 1회 재시도 → 안내 메시지)
- `update_preference` : Store 쓰기, HITL 미들웨어로 보호


In [ ]:
from newslingo.tools import news_search, update_preference, TOOLS
import json

raw = news_search.invoke({'query': 'large language model', 'exclude_urls': []})
articles = json.loads(raw)['articles']
print(len(articles), '건')
for a in articles[:3]:
    print('-', a['title'], '/', a['source'], a['published_date'])

In [ ]:
# update_preference 는 docstring 만 확인 (실제 호출은 Agent + HITL 경유)
print(update_preference.description)

## 5. Middleware (설계서 3.2)

| 미들웨어 | Hook | 종류 | 역할 |
|---|---|---|---|
| `profile_injection` | wrap_model_call | Custom | Store 프로필 → System Prompt |
| `HumanInTheLoopMiddleware` | wrap_tool_call | Built-in | `update_preference` 2차 재확인 |
| `SummarizationMiddleware` | before_model | Built-in | 대화 이력 자동 요약 |
| `ModelFallbackMiddleware` | wrap_model_call | Built-in | 모델1 실패 시 대체 모델 |


In [ ]:
from newslingo.middleware import build_middleware
for m in build_middleware():
    print('-', type(m).__name__ if not callable(getattr(m,'__name__',None)) else m.__class__.__name__)

## 6. Agent 조립 (설계서 2.1)

강의 [4]/[5] 의 `create_agent(model, tools, middleware, checkpointer, store, context_schema)`.


In [ ]:
from newslingo.agent import build_agent, Context
agent = build_agent()
agent

## 7. LCEL 체인 (설계서 2.4)

`prompt | model.with_structured_output(스키마)` — 기사추천 / 학습자료 / 퀴즈 생성.
결정적 품질이 필요해 Agent 가 아니라 체인으로 분리.


In [ ]:
from newslingo.chains import article_recommender_chain, study_material_chain, quiz_chain

cand = article_recommender_chain.invoke({
    'topic': 'large language model', 'level': '중급',
    'seen_urls': '(없음)',
    'raw_articles': json.dumps(articles, ensure_ascii=False),
})
print(type(cand).__name__, '| 기사', len(cand.articles), '개')
for a in cand.articles:
    print('-', a.title, '|', a.keywords)

In [ ]:
study = study_material_chain.invoke({
    'level': '중급',
    'article_title': articles[0]['title'],
    'article_text': articles[0]['content'],
})
print('[번역]', study.translated_text[:120], '...')
print('[전문용어]', [t.term for t in study.key_terms])
print('[문법]', [g.pattern for g in study.grammar_points])

## 8. 채점 & 난이도 추천 로직 (설계서 2.2, 10~11단계)

LLM 없이 순수 파이썬. 실제 난이도 변경은 여기서 하지 않음 (HITL 승인 후).


In [ ]:
from newslingo.grading import grade_quiz, recommend_level

for acc in [1.0, 0.8, 0.6, 0.4, 0.2]:
    rec = recommend_level(acc, '중급')
    print(f'정답률 {acc:.0%} → {rec.direction:4s} ({rec.suggested_level})  {rec.message}')

## 9. 전체 플로우 — `LearningSession` (설계서 2.2)

UI 는 이 클래스 하나만 씁니다. 설계서 2.2 의 1~15단계가 메서드로 매핑돼 있습니다.


In [ ]:
from newslingo.service import LearningSession

s = LearningSession(user_id='demo_user')
s.set_initial_level('중급')                       # 0. 온보딩
print('프로필:', s.profile)

In [ ]:
guard = s.request_topic('거대언어모델 관련해서 공부하고 싶어')   # 1~2. 가드레일
assert guard.allowed, guard
cands = s.recommend_articles(guard.extracted_topic)             # 3~4. 기사 5개
for i, a in enumerate(cands.articles, 1):
    print(f'[{i}] {a.title}')

In [ ]:
s.select_article(cands.articles[0])              # 5. 선택
mat = s.make_study_material()                    # 6. 학습자료
print(mat.translated_text[:150], '...')

In [ ]:
print(s.chat('이 기사에서 제일 중요한 문장 하나만 골라줘'))   # 7. 채팅

In [ ]:
quiz = s.make_quiz()                              # 8~9. 퀴즈 5문항
for i, q in enumerate(quiz.questions, 1):
    print(f'Q{i}. {q.question}')
    for c in q.choices: print('   -', c)

In [ ]:
# 10~11. 채점 + 난이도 추천 (전부 정답이라고 가정)
answers = [q.answer for q in quiz.questions]
result, rec = s.grade(answers)
print(f'{result.correct}/{result.total}  정답률 {result.accuracy:.0%}')
print('추천:', rec.message)

In [ ]:
# 12~14. 난이도 조정: 1차 선택 → HITL 2차 재확인 → 승인
if rec.is_change():
    out = s.propose_level_change(rec)     # 1차 (버튼 클릭 상당)
    print('HITL interrupt:', out['interrupt'])
    out = s.confirm_preference(approve=True)   # 2차 재확인 승인
    print('결과:', out['reply'])
print('최종 프로필:', s.profile)